# Notebook 08a — Phase B: Pre-Registered Health-Flag Thresholds

## Problem

In Notebook 07d, the health-flag signal thresholds were chosen by inspection:
- `T_GREEN_LO = 0.05, T_GREEN_HI = 0.95`
- `T_RED_LO = 0.001, T_RED_HI = 0.999`
- `CLIFF_GREEN_HI = 0.05, CLIFF_RED_LO = 0.20`
- `N_GREEN_LO = 100, N_RED_HI = 30`

These were sensible defaults but **not pre-registered**, and the same data the flag is evaluated on was used implicitly when choosing them. A Q1 reviewer can dismiss the C2 (health flag) contribution as in-sample threshold tuning.

## Phase B fix

Carve the Mondrian holdout (from Phase A) into two further slices, 50/50, class-aware:
- `X_threshold_calibration` — grid-search the three GREEN-edge thresholds to maximize F1
  - Positive class = "misclassified by model"
  - Prediction = "flag is non-GREEN (AMBER or RED)"
- `X_threshold_evaluation` — frozen-threshold evaluation, untouched until thresholds locked

The RED edges (`T_RED_HI=0.999`, `T_RED_LO=0.001`, `CLIFF_RED_LO=0.20`, `N_RED_HI=30`) stay at v2 boundary defaults — they're already at sensible degenerate-case detection levels.

**Pre-registered grid (DO NOT CHANGE after looking at outcomes):**
- `T_GREEN_LO ∈ {0.001, 0.01, 0.05, 0.1, 0.2, 0.3}` (6 values)
- `CLIFF_GREEN_HI ∈ {0.01, 0.05, 0.10, 0.20, 0.30}` (5 values)
- `N_GREEN_LO ∈ {30, 50, 100, 200, 500}` (5 values)

Total: 150 threshold combinations per dataset. The combo maximizing per-sample F1 on the pooled threshold-calibration partition is selected as the **single global threshold tuple** applied across all (dataset, model, predicted-class) cells.

## Outputs

- `calibrators/{ds}/X_threshold_calibration_indices.npy` — sub-slice for tuning
- `calibrators/{ds}/X_threshold_evaluation_indices.npy` — sub-slice for evaluation
- `results/tables/phase_b_threshold_grid_search.csv` — full grid (150 rows × F1 per dataset and overall)
- `results/tables/phase_b_selected_thresholds.json` — locked thresholds
- `results/tables/scts_v2_calib_health_phaseb.csv` — re-evaluated health flag with pre-registered thresholds
- `results/tables/scts_v2_canonical_with_health_phaseb.csv` — per-sample SCTS + Phase B flag
- `results/tables/phase_b_v2_vs_strict_vs_phaseb.csv` — three-way diff
- `results/tables/bootstrap_cis_phaseb.csv` — bootstrap CIs on Phase B headline numbers
- `docs/phase_b_findings.md` — short writeup


In [1]:
# Setup
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)
print(f'Ready: {os.getcwd()}')

Mounted at /content/drive
Ready: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
# Imports and pre-registered constants
import numpy as np
import pandas as pd
import json, joblib, time
from pathlib import Path
from datetime import datetime
from itertools import product
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DATASETS = ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']
ARCHITECTURES = ['rf', 'xgb', 'dnn']
VARIANTS = ['5class_cw', '5class_smote']
MODELS_PER_DATASET = [f'{a}_{v}' for v in VARIANTS for a in ARCHITECTURES]
CLASS_NAMES_5 = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']

# Sub-split of Mondrian holdout
SUB_HOLDOUT_FRAC = 0.50    # 50/50 split of Mondrian holdout
RARE_SUB_THRESHOLD = 10    # classes below this stay 100% in threshold_calibration

# Pre-registered grid (THIS IS THE LOCKED GRID — do not edit after running)
GRID_T_GREEN_LO = [0.001, 0.01, 0.05, 0.1, 0.2, 0.3]
GRID_CLIFF_GREEN_HI = [0.01, 0.05, 0.10, 0.20, 0.30]
GRID_N_GREEN_LO = [30, 50, 100, 200, 500]

# Fixed RED edges (NOT tuned)
T_GREEN_HI_FIXED = 0.95
T_RED_HI = 0.999
T_RED_LO = 0.001
CLIFF_RED_LO = 0.20
CLIFF_THRESH = 0.95
N_RED_HI = 30

# For Mondrian + c3 recomputation
ALPHA_PRIMARY = 0.05
MIN_CALIB_MONDRIAN = 30
EPS = 1e-6

# Bootstrap
N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 42

print(f'Pre-registered grid: {len(GRID_T_GREEN_LO) * len(GRID_CLIFF_GREEN_HI) * len(GRID_N_GREEN_LO)} combinations')
print(f'  T_GREEN_LO: {GRID_T_GREEN_LO}')
print(f'  CLIFF_GREEN_HI: {GRID_CLIFF_GREEN_HI}')
print(f'  N_GREEN_LO: {GRID_N_GREEN_LO}')
print(f'Fixed (NOT tuned): T_RED_HI={T_RED_HI}, T_RED_LO={T_RED_LO}, CLIFF_RED_LO={CLIFF_RED_LO}, N_RED_HI={N_RED_HI}')

Pre-registered grid: 150 combinations
  T_GREEN_LO: [0.001, 0.01, 0.05, 0.1, 0.2, 0.3]
  CLIFF_GREEN_HI: [0.01, 0.05, 0.1, 0.2, 0.3]
  N_GREEN_LO: [30, 50, 100, 200, 500]
Fixed (NOT tuned): T_RED_HI=0.999, T_RED_LO=0.001, CLIFF_RED_LO=0.2, N_RED_HI=30


In [3]:
# Path resolver + conformal helpers (carried from Phase A)
def find_proba_file(dataset, model_name, split):
    fname = f'{model_name}_{split}_proba.npy'
    for subdir in ['probabilities', 'predictions']:
        p = Path(REPO) / 'models' / dataset / subdir / fname
        if p.exists():
            return p
    raise FileNotFoundError(f'No {fname} for {dataset}/{model_name}')

def split_conformal_threshold(probs, y_true, alpha):
    n = len(y_true)
    scores = 1.0 - probs[np.arange(n), y_true]
    q_level = min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)
    return float(np.quantile(scores, q_level))

def mondrian_conformal_thresholds(probs, y_true, alpha, n_classes=5, min_calib=30):
    y_pred = probs.argmax(axis=1)
    marginal = split_conformal_threshold(probs, y_true, alpha)
    thresholds, fallback, n_per_class = {}, [], {}
    for c in range(n_classes):
        mask = (y_pred == c)
        n_c = int(mask.sum())
        n_per_class[c] = n_c
        if n_c < min_calib:
            thresholds[c] = marginal
            fallback.append(c)
        else:
            scores_c = 1.0 - probs[mask, :][np.arange(n_c), y_true[mask]]
            q = min(np.ceil((n_c + 1) * (1 - alpha)) / n_c, 1.0)
            thresholds[c] = float(np.quantile(scores_c, q))
    return thresholds, fallback, n_per_class

def component_3_safe(probs, y_pred, thresholds, eps=1e-9):
    n = len(y_pred)
    sample_thresh = np.array([thresholds[int(p)] for p in y_pred], dtype=np.float64)
    s = 1.0 - probs[np.arange(n), y_pred]
    c3 = np.zeros(n, dtype=np.float32)
    nonzero = sample_thresh > eps
    c3[nonzero] = np.clip(1.0 - s[nonzero] / sample_thresh[nonzero], 0.0, 1.0)
    zero_mask = ~nonzero
    c3[zero_mask] = (s[zero_mask] <= eps).astype(np.float32)
    return c3

print('Helpers ready.')

Helpers ready.


In [4]:
# Load Phase A artifacts (Mondrian holdout indices + strict calibrated probs)
mondrian_holdout_idx = {}
strict_holdout_probs = {}     # for s2 cliff computation
strict_test_probs = {}        # for canonical evaluation

for ds in DATASETS:
    cal_dir = Path(REPO) / 'calibrators' / ds
    mondrian_holdout_idx[ds] = np.load(cal_dir / 'X_mondrian_holdout_indices.npy')
    for model_name in MODELS_PER_DATASET:
        strict_holdout_probs[(ds, model_name)] = np.load(cal_dir / f'{model_name}_holdout_proba_strict.npy')
        strict_test_probs[(ds, model_name)] = np.load(cal_dir / f'{model_name}_test_proba_strict.npy')

print(f'Loaded Mondrian holdout indices for {len(mondrian_holdout_idx)} datasets')
print(f'Loaded strict calibrated probs for {len(strict_holdout_probs)} cells')
for ds in DATASETS:
    print(f'  {ds}: Mondrian holdout size = {len(mondrian_holdout_idx[ds])}')

Loaded Mondrian holdout indices for 3 datasets
Loaded strict calibrated probs for 18 cells
  nsl_kdd_v2: Mondrian holdout size = 5037
  unsw_nb15_v2: Mondrian holdout size = 5415
  cic_ids2017_v2: Mondrian holdout size = 8000


In [5]:
# Sub-split Mondrian holdout (50/50, class-aware)
sub_split_indices = {}   # ds -> {'tcal_idx_within_holdout', 'teval_idx_within_holdout'}

for ds in DATASETS:
    holdout_idx = mondrian_holdout_idx[ds]
    y_calib_full = np.load(f'{REPO}/data/processed/{ds}/y_calib_5class.npy')
    y_holdout = y_calib_full[holdout_idx]
    n = len(y_holdout)
    rng = np.random.RandomState(SEED + 1)  # different seed than Phase A to avoid coupling

    class_counts = pd.Series(y_holdout).value_counts().to_dict()
    rare_sub = [c for c, cnt in class_counts.items() if cnt < RARE_SUB_THRESHOLD]
    common_sub = [c for c, cnt in class_counts.items() if cnt >= RARE_SUB_THRESHOLD]

    tcal_mask = np.zeros(n, dtype=bool)
    for c in rare_sub:
        tcal_mask[y_holdout == c] = True
    for c in common_sub:
        idx_c = np.where(y_holdout == c)[0]
        rng.shuffle(idx_c)
        n_tcal = int(round(len(idx_c) * SUB_HOLDOUT_FRAC))
        tcal_mask[idx_c[:n_tcal]] = True

    tcal_idx_within = np.where(tcal_mask)[0]
    teval_idx_within = np.where(~tcal_mask)[0]

    sub_split_indices[ds] = {
        'tcal_idx_within_holdout': tcal_idx_within,
        'teval_idx_within_holdout': teval_idx_within,
        'rare_sub_classes': rare_sub,
    }

    np.save(Path(REPO) / 'calibrators' / ds / 'X_threshold_calibration_indices.npy', tcal_idx_within)
    np.save(Path(REPO) / 'calibrators' / ds / 'X_threshold_evaluation_indices.npy', teval_idx_within)

    print(f'\n=== {ds} ===')
    print(f'  Mondrian holdout total: {n}')
    print(f'  Threshold-calibration: {len(tcal_idx_within)}')
    print(f'  Threshold-evaluation: {len(teval_idx_within)}')
    print(f'  Rare classes all-in-tcal: {[CLASS_NAMES_5[c] for c in rare_sub]} (counts: {[class_counts[c] for c in rare_sub]})')
    for c in range(5):
        nt = int((y_holdout[tcal_idx_within] == c).sum())
        ne = int((y_holdout[teval_idx_within] == c).sum())
        print(f'    {CLASS_NAMES_5[c]:8s}: tcal={nt:>5d}, teval={ne:>5d}')


=== nsl_kdd_v2 ===
  Mondrian holdout total: 5037
  Threshold-calibration: 2518
  Threshold-evaluation: 2519
  Rare classes all-in-tcal: [] (counts: [])
    Normal  : tcal= 1347, teval= 1347
    DoS     : tcal=  918, teval=  919
    Probe   : tcal=  233, teval=  233
    R2L     : tcal=   20, teval=   20
    U2R     : tcal=    0, teval=    0

=== unsw_nb15_v2 ===
  Mondrian holdout total: 5415
  Threshold-calibration: 2708
  Threshold-evaluation: 2707
  Rare classes all-in-tcal: [] (counts: [])
    Normal  : tcal= 1120, teval= 1120
    DoS     : tcal=  246, teval=  245
    Probe   : tcal=  250, teval=  250
    R2L     : tcal= 1066, teval= 1067
    U2R     : tcal=   26, teval=   25

=== cic_ids2017_v2 ===
  Mondrian holdout total: 8000
  Threshold-calibration: 4000
  Threshold-evaluation: 4000
  Rare classes all-in-tcal: [] (counts: [])
    Normal  : tcal= 3215, teval= 3215
    DoS     : tcal=  538, teval=  537
    Probe   : tcal=  225, teval=  225
    R2L     : tcal=   22, teval=   23


In [6]:
# Build a pooled per-sample dataset for the grid search.
# For each (ds, model, sample_in_tcal): record
#   - mondrian_threshold for the sample's predicted class (uses Phase A thresholds, NOT recomputed per grid)
#   - cliff_fraction for the sample's predicted class (computed on threshold-calibration partition)
#   - n_calib for the sample's predicted class (computed on threshold-calibration partition)
#   - whether the sample was misclassified
# Then for each grid combo, we compute the flag and the F1 against misclassification.

print('Building pooled grid-search data...')
t0 = time.time()
pool_records = []

for ds in DATASETS:
    holdout_idx = mondrian_holdout_idx[ds]
    tcal_idx_within = sub_split_indices[ds]['tcal_idx_within_holdout']
    y_calib_full = np.load(f'{REPO}/data/processed/{ds}/y_calib_5class.npy')
    y_holdout = y_calib_full[holdout_idx]
    y_tcal = y_holdout[tcal_idx_within]

    for model_name in MODELS_PER_DATASET:
        p_holdout = strict_holdout_probs[(ds, model_name)]
        p_tcal = p_holdout[tcal_idx_within]
        y_pred_tcal = p_tcal.argmax(axis=1)
        misclassified = (y_pred_tcal != y_tcal).astype(int)

        # Mondrian thresholds (carried from Phase A on holdout) — we want to use these so the threshold tuning
        # is INDEPENDENT of the conformal threshold tuning
        mthresh, fb, n_per = mondrian_conformal_thresholds(
            p_holdout, y_holdout, ALPHA_PRIMARY, 5, MIN_CALIB_MONDRIAN
        )

        # Cliff fractions on threshold-calibration partition (per predicted class)
        scores_tcal = 1.0 - p_tcal[np.arange(len(y_tcal)), y_tcal]
        cliff_per_class = {}
        n_per_pred_class_tcal = {}
        for pc in range(5):
            mask = y_pred_tcal == pc
            n_in = int(mask.sum())
            n_per_pred_class_tcal[pc] = n_in
            cliff_per_class[pc] = float((scores_tcal[mask] >= CLIFF_THRESH).mean()) if n_in > 0 else float('nan')

        # Per-sample records
        for i in range(len(y_tcal)):
            pc = int(y_pred_tcal[i])
            pool_records.append({
                'dataset': ds,
                'model': model_name,
                'pred_class': pc,
                'true_class': int(y_tcal[i]),
                'misclassified': int(misclassified[i]),
                'mondrian_threshold': float(mthresh[pc]),
                'cliff_fraction': float(cliff_per_class[pc]) if not np.isnan(cliff_per_class[pc]) else 1.0,
                'n_calib': int(n_per_pred_class_tcal[pc]),
            })

df_pool = pd.DataFrame(pool_records)
print(f'\nPooled dataset built in {(time.time()-t0):.1f}s')
print(f'  Total rows: {len(df_pool)}')
print(f'  Per dataset: {df_pool.groupby("dataset").size().to_dict()}')
print(f'  Misclassified rate per dataset:')
for ds in DATASETS:
    sub = df_pool[df_pool['dataset'] == ds]
    print(f'    {ds}: {sub["misclassified"].mean()*100:.1f}% of {len(sub)} samples')

Building pooled grid-search data...

Pooled dataset built in 0.4s
  Total rows: 55356
  Per dataset: {'cic_ids2017_v2': 24000, 'nsl_kdd_v2': 15108, 'unsw_nb15_v2': 16248}
  Misclassified rate per dataset:
    nsl_kdd_v2: 0.3% of 15108 samples
    unsw_nb15_v2: 20.0% of 16248 samples
    cic_ids2017_v2: 1.2% of 24000 samples


In [7]:
# Grid search over (T_GREEN_LO, CLIFF_GREEN_HI, N_GREEN_LO)
# Objective: per-sample F1 where positive = misclassified, prediction = non-GREEN flag

def evaluate_flag_combo(df, t_green_lo, cliff_green_hi, n_green_lo):
    """Apply the flag rules to df and return (precision, recall, f1, n_non_green, n_misclassified)."""
    # Signal 1: threshold zone
    s1 = np.where(
        (df['mondrian_threshold'] >= T_RED_HI) | (df['mondrian_threshold'] < T_RED_LO),
        'red',
        np.where(
            (df['mondrian_threshold'] >= T_GREEN_HI_FIXED) | (df['mondrian_threshold'] <= t_green_lo),
            'amber', 'green'
        )
    )
    # Signal 2: cliff
    cf = df['cliff_fraction'].fillna(1.0).values
    s2 = np.where(cf >= CLIFF_RED_LO, 'red',
                  np.where(cf >= cliff_green_hi, 'amber', 'green'))
    # Signal 3: support
    nc = df['n_calib'].values
    s3 = np.where(nc < N_RED_HI, 'red',
                  np.where(nc < n_green_lo, 'amber', 'green'))

    flags = np.where((s1 == 'red') | (s2 == 'red') | (s3 == 'red'), 'red',
                     np.where((s1 == 'amber') | (s2 == 'amber') | (s3 == 'amber'), 'amber', 'green'))

    non_green = (flags != 'green').astype(int)
    misclass = df['misclassified'].values

    tp = int(((non_green == 1) & (misclass == 1)).sum())
    fp = int(((non_green == 1) & (misclass == 0)).sum())
    fn = int(((non_green == 0) & (misclass == 1)).sum())
    tn = int(((non_green == 0) & (misclass == 0)).sum())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1, tp, fp, fn, tn, flags

print('Running grid search...')
t0 = time.time()
grid_records = []
total_combos = len(GRID_T_GREEN_LO) * len(GRID_CLIFF_GREEN_HI) * len(GRID_N_GREEN_LO)
combo_n = 0

for t_green_lo, cliff_green_hi, n_green_lo in product(GRID_T_GREEN_LO, GRID_CLIFF_GREEN_HI, GRID_N_GREEN_LO):
    combo_n += 1
    # Pooled F1 (per-sample, all datasets combined)
    p, r, f1, tp, fp, fn, tn, _ = evaluate_flag_combo(df_pool, t_green_lo, cliff_green_hi, n_green_lo)
    rec = {
        'T_GREEN_LO': t_green_lo,
        'CLIFF_GREEN_HI': cliff_green_hi,
        'N_GREEN_LO': n_green_lo,
        'pooled_precision': p,
        'pooled_recall': r,
        'pooled_f1': f1,
        'pooled_tp': tp,
        'pooled_fp': fp,
        'pooled_fn': fn,
        'pooled_tn': tn,
    }
    # Per-dataset F1 (sensitivity check)
    for ds in DATASETS:
        sub = df_pool[df_pool['dataset'] == ds]
        p_d, r_d, f1_d, *_ = evaluate_flag_combo(sub, t_green_lo, cliff_green_hi, n_green_lo)
        rec[f'{ds}_precision'] = p_d
        rec[f'{ds}_recall'] = r_d
        rec[f'{ds}_f1'] = f1_d
    grid_records.append(rec)

df_grid = pd.DataFrame(grid_records)
out_dir = Path(REPO) / 'results' / 'tables'
df_grid.to_csv(out_dir / 'phase_b_threshold_grid_search.csv', index=False)
print(f'Grid search complete: {len(df_grid)} combos in {(time.time()-t0):.1f}s')

# Identify best combo by pooled F1
best_idx = df_grid['pooled_f1'].idxmax()
best = df_grid.iloc[best_idx]
print(f'\nBEST COMBO (pooled F1):')
print(f'  T_GREEN_LO = {best["T_GREEN_LO"]}')
print(f'  CLIFF_GREEN_HI = {best["CLIFF_GREEN_HI"]}')
print(f'  N_GREEN_LO = {best["N_GREEN_LO"]}')
print(f'  Pooled F1 = {best["pooled_f1"]:.4f}')
print(f'  Pooled precision = {best["pooled_precision"]:.4f}')
print(f'  Pooled recall = {best["pooled_recall"]:.4f}')
print(f'  Per-dataset F1: NSL={best["nsl_kdd_v2_f1"]:.3f}, UNSW={best["unsw_nb15_v2_f1"]:.3f}, CIC={best["cic_ids2017_v2_f1"]:.3f}')

# Top 5 combos to see if best is meaningfully better than runners-up
print(f'\nTop 5 combos by pooled F1:')
print(df_grid.nlargest(5, 'pooled_f1')[['T_GREEN_LO', 'CLIFF_GREEN_HI', 'N_GREEN_LO',
                                          'pooled_precision', 'pooled_recall', 'pooled_f1']].to_string(index=False))

Running grid search...
Grid search complete: 150 combos in 6.0s

BEST COMBO (pooled F1):
  T_GREEN_LO = 0.001
  CLIFF_GREEN_HI = 0.01
  N_GREEN_LO = 500.0
  Pooled F1 = 0.0243
  Pooled precision = 0.0138
  Pooled recall = 0.1056
  Per-dataset F1: NSL=0.005, UNSW=0.095, CIC=0.015

Top 5 combos by pooled F1:
 T_GREEN_LO  CLIFF_GREEN_HI  N_GREEN_LO  pooled_precision  pooled_recall  pooled_f1
      0.001            0.01         500          0.013752        0.10563   0.024336
      0.001            0.05         500          0.013752        0.10563   0.024336
      0.001            0.10         500          0.013752        0.10563   0.024336
      0.001            0.20         500          0.013752        0.10563   0.024336
      0.001            0.30         500          0.013752        0.10563   0.024336


In [8]:
# Lock the selected thresholds. After this cell, NO further tuning is allowed.
PHASE_B_THRESHOLDS = {
    'T_GREEN_LO': float(best['T_GREEN_LO']),
    'T_GREEN_HI': T_GREEN_HI_FIXED,
    'T_RED_LO': T_RED_LO,
    'T_RED_HI': T_RED_HI,
    'CLIFF_GREEN_HI': float(best['CLIFF_GREEN_HI']),
    'CLIFF_RED_LO': CLIFF_RED_LO,
    'CLIFF_THRESH': CLIFF_THRESH,
    'N_GREEN_LO': int(best['N_GREEN_LO']),
    'N_RED_HI': N_RED_HI,
    'selected_by': 'pooled per-sample F1 on threshold-calibration partition (50% of Mondrian holdout)',
    'grid_size': 150,
    'pooled_f1_on_tcal': float(best['pooled_f1']),
    'tcal_sample_count': int(len(df_pool)),
    'selected_at': datetime.now().isoformat(),
}

with open(out_dir / 'phase_b_selected_thresholds.json', 'w') as f:
    json.dump(PHASE_B_THRESHOLDS, f, indent=2)
print(f'Phase B thresholds locked and saved.')
print(json.dumps(PHASE_B_THRESHOLDS, indent=2))

Phase B thresholds locked and saved.
{
  "T_GREEN_LO": 0.001,
  "T_GREEN_HI": 0.95,
  "T_RED_LO": 0.001,
  "T_RED_HI": 0.999,
  "CLIFF_GREEN_HI": 0.01,
  "CLIFF_RED_LO": 0.2,
  "CLIFF_THRESH": 0.95,
  "N_GREEN_LO": 500,
  "N_RED_HI": 30,
  "selected_by": "pooled per-sample F1 on threshold-calibration partition (50% of Mondrian holdout)",
  "grid_size": 150,
  "pooled_f1_on_tcal": 0.02433621215526375,
  "tcal_sample_count": 55356,
  "selected_at": "2026-06-09T04:19:59.871758"
}


In [9]:
# Sanity check 1: evaluate the locked thresholds on the held-out threshold-evaluation partition
# This is the unbiased estimate of the flag's F1 (still on the Mondrian holdout, but on the half we didn't tune on)

print('=' * 70)
print('SANITY CHECK 1: F1 on threshold-evaluation partition (frozen thresholds)')
print('=' * 70)

teval_records = []
for ds in DATASETS:
    holdout_idx = mondrian_holdout_idx[ds]
    teval_idx_within = sub_split_indices[ds]['teval_idx_within_holdout']
    y_calib_full = np.load(f'{REPO}/data/processed/{ds}/y_calib_5class.npy')
    y_holdout = y_calib_full[holdout_idx]
    y_teval = y_holdout[teval_idx_within]

    for model_name in MODELS_PER_DATASET:
        p_holdout = strict_holdout_probs[(ds, model_name)]
        p_teval = p_holdout[teval_idx_within]
        y_pred_teval = p_teval.argmax(axis=1)
        misclassified = (y_pred_teval != y_teval).astype(int)

        mthresh, _, _ = mondrian_conformal_thresholds(p_holdout, y_holdout, ALPHA_PRIMARY, 5, MIN_CALIB_MONDRIAN)
        scores_teval = 1.0 - p_teval[np.arange(len(y_teval)), y_teval]
        cliff_per_class = {}
        n_per_pred = {}
        for pc in range(5):
            mask = y_pred_teval == pc
            n_in = int(mask.sum())
            n_per_pred[pc] = n_in
            cliff_per_class[pc] = float((scores_teval[mask] >= CLIFF_THRESH).mean()) if n_in > 0 else 1.0

        for i in range(len(y_teval)):
            pc = int(y_pred_teval[i])
            teval_records.append({
                'dataset': ds, 'model': model_name,
                'pred_class': pc, 'misclassified': int(misclassified[i]),
                'mondrian_threshold': float(mthresh[pc]),
                'cliff_fraction': float(cliff_per_class[pc]),
                'n_calib': int(n_per_pred[pc]),
            })

df_teval = pd.DataFrame(teval_records)
p, r, f1, tp, fp, fn, tn, _ = evaluate_flag_combo(
    df_teval,
    PHASE_B_THRESHOLDS['T_GREEN_LO'],
    PHASE_B_THRESHOLDS['CLIFF_GREEN_HI'],
    PHASE_B_THRESHOLDS['N_GREEN_LO'],
)
print(f'Pooled F1 on threshold-evaluation: {f1:.4f}  (precision={p:.4f}, recall={r:.4f})')
print(f'Compare to threshold-calibration F1: {best["pooled_f1"]:.4f}')
print(f'Delta: {f1 - best["pooled_f1"]:+.4f}  (small delta = robust selection; large delta = overfit to tcal)')

# Per-dataset
print(f'\nPer-dataset F1 on threshold-evaluation:')
for ds in DATASETS:
    sub = df_teval[df_teval['dataset'] == ds]
    p_d, r_d, f1_d, *_ = evaluate_flag_combo(
        sub,
        PHASE_B_THRESHOLDS['T_GREEN_LO'],
        PHASE_B_THRESHOLDS['CLIFF_GREEN_HI'],
        PHASE_B_THRESHOLDS['N_GREEN_LO'],
    )
    f1_tcal_ds = best[f'{ds}_f1']
    print(f'  {ds}: tcal F1={f1_tcal_ds:.4f}, teval F1={f1_d:.4f}, delta={f1_d-f1_tcal_ds:+.4f}')

SANITY CHECK 1: F1 on threshold-evaluation partition (frozen thresholds)
Pooled F1 on threshold-evaluation: 0.0265  (precision=0.0150, recall=0.1150)
Compare to threshold-calibration F1: 0.0243
Delta: +0.0021  (small delta = robust selection; large delta = overfit to tcal)

Per-dataset F1 on threshold-evaluation:
  nsl_kdd_v2: tcal F1=0.0046, teval F1=0.0043, delta=-0.0003
  unsw_nb15_v2: tcal F1=0.0953, teval F1=0.1046, delta=+0.0093
  cic_ids2017_v2: tcal F1=0.0145, teval F1=0.0160, delta=+0.0015


In [10]:
# THE REAL EVALUATION: apply Phase B locked thresholds to the canonical 1000 health-flag computation
# This is where the headline NSL R2L catch rate comes from

print('=' * 70)
print('PHASE B HEALTH FLAG ON CANONICAL 1000')
print('=' * 70)

def flag_threshold_pb(t):
    if t >= PHASE_B_THRESHOLDS['T_RED_HI'] or t < PHASE_B_THRESHOLDS['T_RED_LO']:
        return 'red'
    if t >= PHASE_B_THRESHOLDS['T_GREEN_HI'] or t <= PHASE_B_THRESHOLDS['T_GREEN_LO']:
        return 'amber'
    return 'green'

def flag_cliff_pb(f):
    if np.isnan(f): return 'red'
    if f >= PHASE_B_THRESHOLDS['CLIFF_RED_LO']: return 'red'
    if f >= PHASE_B_THRESHOLDS['CLIFF_GREEN_HI']: return 'amber'
    return 'green'

def flag_support_pb(n):
    if n < PHASE_B_THRESHOLDS['N_RED_HI']: return 'red'
    if n < PHASE_B_THRESHOLDS['N_GREEN_LO']: return 'amber'
    return 'green'

def combine_flags(*flags):
    if 'red' in flags: return 'red'
    if 'amber' in flags: return 'amber'
    return 'green'

# Compute cliff fractions on the FULL Mondrian holdout (consistent with Phase A protocol —
# Phase B only changes the threshold values, not the partition the flag is fit on)
cliff_data = {}
for ds in DATASETS:
    holdout_idx = mondrian_holdout_idx[ds]
    y_calib_full = np.load(f'{REPO}/data/processed/{ds}/y_calib_5class.npy')
    y_holdout = y_calib_full[holdout_idx]
    for model_name in MODELS_PER_DATASET:
        p_holdout = strict_holdout_probs[(ds, model_name)]
        scores = 1.0 - p_holdout[np.arange(len(y_holdout)), y_holdout]
        y_pred_h = p_holdout.argmax(axis=1)
        for pred_cls in range(5):
            mask = y_pred_h == pred_cls
            n_in = int(mask.sum())
            cf = float('nan') if n_in == 0 else float((scores[mask] >= PHASE_B_THRESHOLDS['CLIFF_THRESH']).mean())
            cliff_data[(ds, model_name, pred_cls)] = {'cliff_fraction': cf, 'n_pred_class_in_holdout': n_in}

# Build per-cell health table
health_records_pb = []
conformal_meta_pb = {}
for ds in DATASETS:
    holdout_idx = mondrian_holdout_idx[ds]
    y_calib_full = np.load(f'{REPO}/data/processed/{ds}/y_calib_5class.npy')
    y_holdout = y_calib_full[holdout_idx]
    for model_name in MODELS_PER_DATASET:
        p_holdout = strict_holdout_probs[(ds, model_name)]
        mthresh, fb, n_per = mondrian_conformal_thresholds(p_holdout, y_holdout, ALPHA_PRIMARY, 5, MIN_CALIB_MONDRIAN)
        conformal_meta_pb[f'{ds}/{model_name}'] = {
            'mondrian_thresholds': {str(c): float(t) for c, t in mthresh.items()},
            'fallback_classes': [int(c) for c in fb],
            'n_per_class': {str(c): int(n) for c, n in n_per.items()},
        }
        for pred_cls in range(5):
            t = mthresh[pred_cls]
            cf = cliff_data[(ds, model_name, pred_cls)]['cliff_fraction']
            n_calib_pb = n_per[pred_cls]
            s_t = flag_threshold_pb(t)
            s_c = flag_cliff_pb(cf)
            s_s = flag_support_pb(n_calib_pb)
            overall = combine_flags(s_t, s_c, s_s)
            health_records_pb.append({
                'dataset': ds, 'model': model_name,
                'predicted_class_idx': pred_cls, 'predicted_class': CLASS_NAMES_5[pred_cls],
                'mondrian_threshold': t, 'is_fallback': pred_cls in fb,
                'n_calib': n_calib_pb, 'cliff_fraction': cf,
                'signal_threshold': s_t, 'signal_cliff': s_c, 'signal_support': s_s,
                'calib_health': overall,
            })

df_health_pb = pd.DataFrame(health_records_pb)
df_health_pb.to_csv(out_dir / 'scts_v2_calib_health_phaseb.csv', index=False)

print(f'Phase B health table: {len(df_health_pb)} rows')
print(f'Class-level (PhaseB): {dict(df_health_pb["calib_health"].value_counts())}')
print(f'\nPer-dataset (PhaseB):')
print(df_health_pb.groupby(['dataset', 'calib_health']).size().unstack(fill_value=0))

PHASE B HEALTH FLAG ON CANONICAL 1000
Phase B health table: 90 rows
Class-level (PhaseB): {'red': np.int64(37), 'amber': np.int64(31), 'green': np.int64(22)}

Per-dataset (PhaseB):
calib_health    amber  green  red
dataset                          
cic_ids2017_v2     10     10   10
nsl_kdd_v2          8      1   21
unsw_nb15_v2       13     11    6


In [11]:
# Augment per-sample SCTS with Phase B health flag, then compute headline numbers
df_scts_safe = pd.read_csv(out_dir / 'scts_v2_canonical_strict_safe.csv')
flag_lookup_pb = df_health_pb.set_index(['dataset', 'model', 'predicted_class_idx'])['calib_health'].to_dict()
df_scts_safe['calib_health_phaseb'] = df_scts_safe.apply(
    lambda row: flag_lookup_pb.get((row['dataset'], row['model'], row['pred_class']), 'unknown'),
    axis=1,
)
df_scts_safe.to_csv(out_dir / 'scts_v2_canonical_with_health_phaseb.csv', index=False)

print(f'Sample-level (PhaseB): {dict(df_scts_safe["calib_health_phaseb"].value_counts())}')

# Per-flag mean Pearson
per_flag_pearson_pb = {}
for flag in ['green', 'amber', 'red']:
    sub = df_scts_safe[df_scts_safe['calib_health_phaseb'] == flag]
    per_m = []
    for (ds, m), g in sub.groupby(['dataset', 'model']):
        if g['scts'].std() > 1e-9 and g['correct'].std() > 1e-9 and len(g) > 5:
            p = float(np.corrcoef(g['scts'], g['correct'])[0, 1])
            if not np.isnan(p): per_m.append(p)
    per_flag_pearson_pb[flag] = float(np.mean(per_m)) if per_m else None

print(f'\nPer-flag mean Pearson (PhaseB):')
for f in ['green', 'amber', 'red']:
    p = per_flag_pearson_pb[f]
    print(f'  {f.upper():>5}: {p:+.3f}' if p is not None else f'  {f.upper():>5}: N/A')

# Headline: NSL R2L catch rate
nsl_r2l = df_scts_safe[(df_scts_safe['dataset'] == 'nsl_kdd_v2') & (df_scts_safe['true_class'] == 3)]
red_pb = int((nsl_r2l['calib_health_phaseb'] == 'red').sum())
n_pb = len(nsl_r2l)
print(f'\n*** NSL R2L RED-flag catch rate (PhaseB): {red_pb}/{n_pb} = {100*red_pb/n_pb:.1f}% ***')

# Also: any-non-GREEN catch rate (more practically relevant)
nongreen_pb = int((nsl_r2l['calib_health_phaseb'] != 'green').sum())
print(f'*** NSL R2L non-GREEN flag rate (PhaseB): {nongreen_pb}/{n_pb} = {100*nongreen_pb/n_pb:.1f}% ***')

Sample-level (PhaseB): {'green': np.int64(7346), 'red': np.int64(6365), 'amber': np.int64(4289)}

Per-flag mean Pearson (PhaseB):
  GREEN: +0.418
  AMBER: +0.170
    RED: +0.240

*** NSL R2L RED-flag catch rate (PhaseB): 1007/1278 = 78.8% ***
*** NSL R2L non-GREEN flag rate (PhaseB): 1087/1278 = 85.1% ***


In [12]:
# Bootstrap CIs on Phase B headline metrics
print('=' * 70)
print('BOOTSTRAP CIs ON PHASE B (B=1000)')
print('=' * 70)

rng_boot = np.random.RandomState(BOOTSTRAP_SEED)
boot_records_pb = []

# Per-flag mean Pearson with CIs
for flag in ['green', 'amber', 'red']:
    sub = df_scts_safe[df_scts_safe['calib_health_phaseb'] == flag]
    per_m_p = []
    for (ds, m), g in sub.groupby(['dataset', 'model']):
        if g['scts'].std() > 1e-9 and g['correct'].std() > 1e-9 and len(g) > 5:
            p = float(np.corrcoef(g['scts'], g['correct'])[0, 1])
            if not np.isnan(p): per_m_p.append(p)
    if len(per_m_p) < 2: continue
    arr = np.array(per_m_p)
    n = len(arr)
    boots = np.array([arr[rng_boot.randint(0, n, n)].mean() for _ in range(N_BOOTSTRAP)])
    boot_records_pb.append({
        'protocol': 'phaseb', 'metric': 'per_flag_mean_pearson', 'flag': flag,
        'point': float(arr.mean()),
        'ci_low': float(np.percentile(boots, 2.5)),
        'ci_high': float(np.percentile(boots, 97.5)),
    })

# NSL R2L catch rate (RED only, and any-non-GREEN)
nsl_r2l = df_scts_safe[(df_scts_safe['dataset'] == 'nsl_kdd_v2') & (df_scts_safe['true_class'] == 3)]
is_red = (nsl_r2l['calib_health_phaseb'] == 'red').astype(int).values
is_nongreen = (nsl_r2l['calib_health_phaseb'] != 'green').astype(int).values

for name, arr in [('r2l_red_catch_rate', is_red), ('r2l_nongreen_catch_rate', is_nongreen)]:
    n = len(arr)
    boots = np.array([arr[rng_boot.randint(0, n, n)].mean() for _ in range(N_BOOTSTRAP)])
    k = int(arr.sum())
    p_hat = k / n
    z = 1.96
    denom = 1 + z**2/n
    centre = (p_hat + z**2/(2*n)) / denom
    halfw = z * np.sqrt(p_hat*(1-p_hat)/n + z**2/(4*n*n)) / denom
    boot_records_pb.append({
        'protocol': 'phaseb', 'metric': name, 'flag': 'n/a',
        'point': float(p_hat),
        'ci_low': float(np.percentile(boots, 2.5)),
        'ci_high': float(np.percentile(boots, 97.5)),
        'wilson_ci_low': float(centre - halfw),
        'wilson_ci_high': float(centre + halfw),
        'n_total': int(n), 'n_red': int(k),
    })

df_boot_pb = pd.DataFrame(boot_records_pb)
df_boot_pb.to_csv(out_dir / 'bootstrap_cis_phaseb.csv', index=False)

print(f'Bootstrap records: {len(df_boot_pb)}')
print(f'\nPhase B headline numbers with 95% CIs:')
for _, row in df_boot_pb.iterrows():
    if row['metric'] == 'per_flag_mean_pearson':
        print(f'  {row["flag"].upper()} Pearson: {row["point"]:+.3f} [{row["ci_low"]:+.3f}, {row["ci_high"]:+.3f}]')
    else:
        print(f'  {row["metric"]}: {row["point"]*100:.1f}% bootstrap [{row["ci_low"]*100:.1f}%, {row["ci_high"]*100:.1f}%]')
        print(f'    {" "*30} Wilson    [{row["wilson_ci_low"]*100:.1f}%, {row["wilson_ci_high"]*100:.1f}%]')

BOOTSTRAP CIs ON PHASE B (B=1000)
Bootstrap records: 5

Phase B headline numbers with 95% CIs:
  GREEN Pearson: +0.418 [+0.367, +0.473]
  AMBER Pearson: +0.170 [-0.048, +0.334]
  RED Pearson: +0.240 [+0.131, +0.357]
  r2l_red_catch_rate: 78.8% bootstrap [76.5%, 81.1%]
                                   Wilson    [76.5%, 80.9%]
  r2l_nongreen_catch_rate: 85.1% bootstrap [83.0%, 86.9%]
                                   Wilson    [83.0%, 86.9%]


In [13]:
# Three-way comparison: v2 vs strict-safe vs PhaseB
df_v2_scts = pd.read_csv(out_dir / 'scts_v2_canonical.csv')
df_v2_health = pd.read_csv(out_dir / 'scts_v2_calib_health.csv')

with open(out_dir / 'scts_v2_summary.json') as f:
    v2_sum = json.load(f)
with open(out_dir / 'scts_v2_health_summary.json') as f:
    v2_health_sum = json.load(f)

# NSL R2L catch rates for v2
df_v2_merged = df_v2_scts[(df_v2_scts['dataset'] == 'nsl_kdd_v2') & (df_v2_scts['true_class'] == 3)].merge(
    df_v2_health[['dataset', 'model', 'predicted_class_idx', 'calib_health']],
    left_on=['dataset', 'model', 'pred_class'],
    right_on=['dataset', 'model', 'predicted_class_idx'],
)
n_v2 = len(df_v2_merged)
red_v2 = int((df_v2_merged['calib_health'] == 'red').sum())
nongreen_v2 = int((df_v2_merged['calib_health'] != 'green').sum())

# strict-safe
nsl_r2l_safe = df_scts_safe[(df_scts_safe['dataset'] == 'nsl_kdd_v2') & (df_scts_safe['true_class'] == 3)]
# strict-safe column from 07f. Need to load it from canonical_with_health_strict_safe
df_strict_safe_health = pd.read_csv(out_dir / 'scts_v2_canonical_with_health_strict_safe.csv')
nsl_r2l_strict_safe = df_strict_safe_health[(df_strict_safe_health['dataset'] == 'nsl_kdd_v2') & (df_strict_safe_health['true_class'] == 3)]
red_strict_safe = int((nsl_r2l_strict_safe['calib_health'] == 'red').sum())
nongreen_strict_safe = int((nsl_r2l_strict_safe['calib_health'] != 'green').sum())
n_strict_safe = len(nsl_r2l_strict_safe)

# PhaseB
red_pb = int((nsl_r2l['calib_health_phaseb'] == 'red').sum())
nongreen_pb = int((nsl_r2l['calib_health_phaseb'] != 'green').sum())
n_pb = len(nsl_r2l)

print('=' * 90)
print('THREE-WAY COMPARISON: v2 vs STRICT-SAFE vs PHASEB')
print('=' * 90)
print()
print(f"{'Metric':<48}{'v2':>14}{'strict-safe':>14}{'PhaseB':>14}")
print('-' * 90)

# Class-level flag distribution
print(f'\nClass-level flag distribution:')
strict_safe_health = pd.read_csv(out_dir / 'scts_v2_calib_health_strict_safe.csv')
v2_cl = v2_health_sum['overall_flag_counts']['class_level']
ss_cl = dict(strict_safe_health['calib_health'].value_counts())
pb_cl = dict(df_health_pb['calib_health'].value_counts())
for f in ['green', 'amber', 'red']:
    print(f"  {f:>6}: v2={v2_cl.get(f, 0):>3}  strict-safe={ss_cl.get(f, 0):>3}  phaseB={pb_cl.get(f, 0):>3}")

# Sample-level flag distribution
print(f'\nSample-level flag distribution:')
v2_sl = v2_health_sum['overall_flag_counts']['sample_level']
ss_sl = dict(df_strict_safe_health['calib_health'].value_counts())
pb_sl = dict(df_scts_safe['calib_health_phaseb'].value_counts())
for f in ['green', 'amber', 'red']:
    print(f"  {f:>6}: v2={v2_sl.get(f, 0):>5}  strict-safe={ss_sl.get(f, 0):>5}  phaseB={pb_sl.get(f, 0):>5}")

# Per-flag Pearson
print(f'\nPer-flag mean Pearson:')
ss_p = {}
for flag in ['green', 'amber', 'red']:
    sub = df_strict_safe_health.merge(df_scts_safe[['dataset', 'model', 'sample_position', 'scts', 'correct']],
                                       on=['dataset', 'model'], how='inner')
    # actually re-derive from canonical_with_health_strict_safe
    sub2 = pd.read_csv(out_dir / 'scts_v2_canonical_with_health_strict_safe.csv')
    sub2 = sub2[sub2['calib_health'] == flag]
    per_m = []
    for (ds, m), g in sub2.groupby(['dataset', 'model']):
        if g['scts'].std() > 1e-9 and g['correct'].std() > 1e-9 and len(g) > 5:
            p = float(np.corrcoef(g['scts'], g['correct'])[0, 1])
            if not np.isnan(p): per_m.append(p)
    ss_p[flag] = float(np.mean(per_m)) if per_m else None

for f in ['green', 'amber', 'red']:
    v_p = v2_health_sum['summary_by_flag'][f]['mean_pearson_per_model']
    print(f"  {f.upper():>5}: v2={v_p:+.3f}  strict-safe={ss_p[f]:+.3f}  phaseB={per_flag_pearson_pb[f]:+.3f}")

# Headline NSL R2L
print(f'\n*** NSL R2L RED-flag catch rate (headline) ***')
print(f'  v2:          {red_v2}/{n_v2} = {100*red_v2/n_v2:.1f}%')
print(f'  strict-safe: {red_strict_safe}/{n_strict_safe} = {100*red_strict_safe/n_strict_safe:.1f}%')
print(f'  PhaseB:      {red_pb}/{n_pb} = {100*red_pb/n_pb:.1f}%')
print(f'\n*** NSL R2L non-GREEN catch rate (operational metric) ***')
print(f'  v2:          {nongreen_v2}/{n_v2} = {100*nongreen_v2/n_v2:.1f}%')
print(f'  strict-safe: {nongreen_strict_safe}/{n_strict_safe} = {100*nongreen_strict_safe/n_strict_safe:.1f}%')
print(f'  PhaseB:      {nongreen_pb}/{n_pb} = {100*nongreen_pb/n_pb:.1f}%')

# Save three-way diff
diff_rows = []
for ds in DATASETS:
    for model_name in MODELS_PER_DATASET:
        for pred_cls in range(5):
            v2_match = df_v2_health[(df_v2_health['dataset'] == ds) & (df_v2_health['model'] == model_name) & (df_v2_health['predicted_class_idx'] == pred_cls)]
            ss_match = strict_safe_health[(strict_safe_health['dataset'] == ds) & (strict_safe_health['model'] == model_name) & (strict_safe_health['predicted_class_idx'] == pred_cls)]
            pb_match = df_health_pb[(df_health_pb['dataset'] == ds) & (df_health_pb['model'] == model_name) & (df_health_pb['predicted_class_idx'] == pred_cls)]

            v2_f = v2_match['calib_health'].iloc[0] if len(v2_match) else 'na'
            ss_f = ss_match['calib_health'].iloc[0] if len(ss_match) else 'na'
            pb_f = pb_match['calib_health'].iloc[0] if len(pb_match) else 'na'

            diff_rows.append({
                'dataset': ds, 'model': model_name,
                'predicted_class': CLASS_NAMES_5[pred_cls],
                'flag_v2': v2_f, 'flag_strict_safe': ss_f, 'flag_phaseb': pb_f,
                'changed_v2_to_pb': v2_f != pb_f,
            })

df_diff_3way = pd.DataFrame(diff_rows)
df_diff_3way.to_csv(out_dir / 'phase_b_v2_vs_strict_vs_phaseb.csv', index=False)
print(f'\nSaved phase_b_v2_vs_strict_vs_phaseb.csv ({len(df_diff_3way)} rows)')

THREE-WAY COMPARISON: v2 vs STRICT-SAFE vs PHASEB

Metric                                                      v2   strict-safe        PhaseB
------------------------------------------------------------------------------------------

Class-level flag distribution:
   green: v2= 37  strict-safe= 24  phaseB= 22
   amber: v2= 16  strict-safe= 29  phaseB= 31
     red: v2= 37  strict-safe= 37  phaseB= 37

Sample-level flag distribution:
   green: v2= 7618  strict-safe= 6663  phaseB= 7346
   amber: v2= 3423  strict-safe= 4972  phaseB= 4289
     red: v2= 6959  strict-safe= 6365  phaseB= 6365

Per-flag mean Pearson:
  GREEN: v2=+0.292  strict-safe=+0.432  phaseB=+0.418
  AMBER: v2=+0.351  strict-safe=+0.151  phaseB=+0.170
    RED: v2=+0.184  strict-safe=+0.240  phaseB=+0.240

*** NSL R2L RED-flag catch rate (headline) ***
  v2:          1206/1278 = 94.4%
  strict-safe: 1007/1278 = 78.8%
  PhaseB:      1007/1278 = 78.8%

*** NSL R2L non-GREEN catch rate (operational metric) ***
  v2:          1

In [14]:
# Phase B findings doc
findings_md = f"""# Phase B Findings — Pre-Registered Health-Flag Thresholds

**Date**: {datetime.now().strftime('%Y-%m-%d')}
**Status**: For internal record; supervisor review deferred until all experimental phases complete

## What we tested

The v2 health flag's three GREEN-edge thresholds (T_GREEN_LO=0.05, CLIFF_GREEN_HI=0.05, N_GREEN_LO=100) were chosen by inspection on the same data the flag was evaluated on. Phase B replaces this with pre-registered grid search on a 50% sub-slice of the Phase A Mondrian holdout.

## Selected thresholds

- T_GREEN_LO = {PHASE_B_THRESHOLDS['T_GREEN_LO']}
- CLIFF_GREEN_HI = {PHASE_B_THRESHOLDS['CLIFF_GREEN_HI']}
- N_GREEN_LO = {PHASE_B_THRESHOLDS['N_GREEN_LO']}

Grid size: 150 combinations. Selection criterion: pooled per-sample F1 against misclassification labels on the threshold-calibration partition. Pooled F1 at selected thresholds: {PHASE_B_THRESHOLDS['pooled_f1_on_tcal']:.4f}.

## Headline results

| Metric | v2 | Strict-safe (Phase A) | Phase B (pre-registered) |
|---|---|---|---|
| Class-level flags | G{v2_cl.get('green',0)}/A{v2_cl.get('amber',0)}/R{v2_cl.get('red',0)} | G{ss_cl.get('green',0)}/A{ss_cl.get('amber',0)}/R{ss_cl.get('red',0)} | G{pb_cl.get('green',0)}/A{pb_cl.get('amber',0)}/R{pb_cl.get('red',0)} |
| Sample-level flags | G{v2_sl.get('green',0)}/A{v2_sl.get('amber',0)}/R{v2_sl.get('red',0)} | G{ss_sl.get('green',0)}/A{ss_sl.get('amber',0)}/R{ss_sl.get('red',0)} | G{pb_sl.get('green',0)}/A{pb_sl.get('amber',0)}/R{pb_sl.get('red',0)} |
| NSL R2L RED rate | {100*red_v2/n_v2:.1f}% | {100*red_strict_safe/n_strict_safe:.1f}% | {100*red_pb/n_pb:.1f}% |
| NSL R2L non-GREEN rate | {100*nongreen_v2/n_v2:.1f}% | {100*nongreen_strict_safe/n_strict_safe:.1f}% | {100*nongreen_pb/n_pb:.1f}% |

## Files produced

- `notebooks/08a_phase_b_pre_registered_flags.ipynb`
- `results/tables/phase_b_threshold_grid_search.csv`
- `results/tables/phase_b_selected_thresholds.json`
- `results/tables/scts_v2_calib_health_phaseb.csv`
- `results/tables/scts_v2_canonical_with_health_phaseb.csv`
- `results/tables/phase_b_v2_vs_strict_vs_phaseb.csv`
- `results/tables/bootstrap_cis_phaseb.csv`
"""

docs_dir = Path(REPO) / 'docs'
with open(docs_dir / 'phase_b_findings.md', 'w') as f:
    f.write(findings_md)
print(f'Wrote: docs/phase_b_findings.md')

Wrote: docs/phase_b_findings.md


In [ ]:
# Commit and push
os.chdir(REPO)
!git status --short

print('\n>>> Staging Phase B files...')
!git add notebooks/08a_phase_b_pre_registered_flags.ipynb
!git add calibrators/*/X_threshold_calibration_indices.npy
!git add calibrators/*/X_threshold_evaluation_indices.npy
!git add results/tables/phase_b_threshold_grid_search.csv
!git add results/tables/phase_b_selected_thresholds.json
!git add results/tables/scts_v2_calib_health_phaseb.csv
!git add results/tables/scts_v2_canonical_with_health_phaseb.csv
!git add results/tables/phase_b_v2_vs_strict_vs_phaseb.csv
!git add results/tables/bootstrap_cis_phaseb.csv
!git add docs/phase_b_findings.md

!git status --short
!git commit -m "Phase B: pre-registered health-flag thresholds — 150-combo grid search on threshold-calibration sub-slice of Mondrian holdout, selection by pooled per-sample F1 against misclassification, three-way comparison with v2 + strict-safe"
!git push origin main

print('\n>>> Drive saved + git pushed? Confirm before we move on to Block 2 (multi-seed).')